# Notebook AI Node — Design Spec (Tier 1)

**Date:** 2026-06-03  
**Status:** Approved design, pre-implementation  
**Scope:** Tier 1 (single-shot AI node). Tier 2 (session/Orchestrator-backed node) is a documented evolution path, not buildable scope here.

This spec is authored as a notebook to dogfood the very surface it designs.

## 1. Goal & context

**Goal:** Let the jute-notebook reactive DAG engine host an *AI node* — a cell whose body executes against **Spur** instead of a Python/Deno kernel. It takes upstream port values as context and writes its answer to an output port the rest of the DAG consumes.

**Grounding (from prior code-graph analysis):**

- The DAG engine lives at `crates/spur-notebook/src/dag` (`ReactiveEngine<R: CellRunner>`). Cells declare `produces`/`consumes` ports; the engine cascades re-runs to dependents when an upstream port version changes.
- The pluggable seam is the **`CellRunner`** trait (`engine.rs:108`). The production impl `RunCellCommandRunner::run_cell` forwards `{cell_id, kernel_id, code, …}` to a kernel via `run_cell::call`.
- Ports are **Arrow IPC RecordBatches on disk** (`~/.spur/notebooks/<id>/ports/` + a `manifest.json`), normally read/written by kernels via injected bootstrap. An AI node has no kernel, so its runner touches `PortStore` directly.
- Spur exposes a thin single-turn surface: the **`AgentConnection`** trait (`crates/spur-acp/src/connection/mod.rs`) — `initialize()` → `new_session(cwd, mcp_servers)` → `prompt(req) -> Stream<SessionNotification>` (turn completes when the stream closes) — with adapters (native ACP, cli-wrap, stdio, stream-json). **No `Orchestrator` required.**
- `spur-notebook` already depends on `spur-acp` and `spur-core` (Cargo.toml), so Tier 1 adds **no new heavy dependency**. The expensive `Orchestrator` lift is deferred to Tier 2.

## 2. Decisions (from brainstorming)

1. **Tiered.** Single-shot AI node now (Tier 1); session/Orchestrator-backed node later (Tier 2), behind one trait.
2. **Authoring contract.** Prompt-as-body, ports-as-context, prompt → text → output port. Structured/schema output deferred to Tier 1.5.
3. **Execution policy.** Per-node policy, **default manual + stale-marking**; input-hash caching always on; opt-in live mode.
4. **Wiring.** Approach 1 — thin `AgentConnection` client, behind an `AiNodeBackend` trait so Tier 2 plugs in with no engine change.

## 3. Architecture — three new units

### 3.1 `AiNodeBackend` trait (new `crates/spur-notebook/src/dag/ai/mod.rs`)
The only AI abstraction the engine knows about — the Tier-1/Tier-2 seam.

```rust
#[async_trait]
pub trait AiNodeBackend: Send + Sync {
    async fn run(&self, req: AiRunRequest) -> Result<AiRunOutput, AiError>;
}

pub struct AiRunRequest {
    pub cell_id: String,
    pub prompt: String,            // the cell body
    pub context: Vec<PortContext>, // rendered consumed ports
    pub cancel: CancellationToken,
}
pub struct AiRunOutput { pub text: String, pub usage: Option<AiUsage> }
```

### 3.2 `AcpAgentBackend` (Tier-1 impl, `.../ai/acp_backend.rs`)
Wraps `Arc<Mutex<dyn AgentConnection>>` from **spur-acp**. Lazily `initialize()` + `new_session(cwd, mcp_servers)` once per notebook. Per run: build a `PromptRequest` from `prompt + context`, call `connection.prompt(...)`, and **drain the `SessionNotification` stream into final assistant text** (stream close = turn complete). Owns nothing the `Orchestrator` owns.

### 3.3 `NotebookCellRunner` (composite `CellRunner`)
Wraps the existing `RunCellCommandRunner` **and** `Arc<dyn AiNodeBackend>`. In `run_cell`, classify the cell by kernelspec (reusing the existing kernelspec-routing signal): a normal kernelspec → existing `run_cell::call` path unchanged; the **`spur` kernelspec → AI path**.

## 4. Cell model (authoring)

An AI node is a cell with kernelspec **`spur`**. **Body = the prompt.** It declares `consumes` (upstream ports injected as context) and one produced **text port** (the answer), exactly like any reactive cell. **No new notebook schema** — reuses existing cell metadata + ports.

## 5. Data flow (one AI-node run)

```
engine.run_cell(req)                         // existing cascade entry
  └─ NotebookCellRunner.run_cell(req)
       ├─ kernelspec == "spur"? ── no ──▶ RunCellCommandRunner (unchanged)
       └─ yes:
            1. resolve cell's consumes/produces port names (notebook metadata)
            2. PortStore.read(consumed) → render Arrow batches → text/JSON context
            3. AiNodeBackend.run({ prompt: body, context })
                 // AcpAgentBackend → connection.prompt → drain stream → text
            4. PortStore.write(produced_text_port, RecordBatch[utf8 col], version+1)
            5. return CellRunOutcome { Succeeded }   // engine cascades downstream
```

The engine's reactive cascade, versioning, and downstream propagation are untouched. The AI node is just a cell whose body executes against Spur instead of a kernel, reading/writing the same Arrow ports.

## 6. Execution policy (cost / nondeterminism control)

- **Default = manual + stale-marking.** When an upstream port version changes, the engine marks the AI node **stale** instead of cascading a run into it. Its output port still propagates downstream once the user runs it. This requires the single engine touch: a `CellRunStatus::Stale` (surfaced through the existing `run_cell_with_status_events`) and a cascade rule: *AI node in manual mode → mark stale, do not execute.*
- **Input-hash cache, always on.** Key = `hash(prompt_body + [(consumed_port, version)] + backend_id)`. A run whose key matches the last successful run rewrites the output port from cache with **no Spur call**.
- **Opt-in live mode.** Cell-metadata flag `ai_live: true` makes the node cascade like a normal reactive cell. Cache still applies, so identical inputs never re-bill.

## 7. Error & cancellation handling

- `AiError` is mapped to `EngineError` (same pattern as the existing `StaleCell`/`RunCell` mapping) so a failed node shows `Failed` with an actionable message. Variants: connection/init failure, prompt-turn error, timeout, cancelled, port read/write, and `no produced text port declared`.
- **Health / re-init:** an unhealthy `AgentConnection` fails fast; the session lazily re-initializes on the next run.
- **Timeout + cancel:** bounded prompt turn; UI/engine cancel → `CancellationToken` → `connection.cancel(session_id)` and stop draining the stream.

## 8. Testing

- `AcpAgentBackend` against spur-acp's existing `TestStubConnection` / `NullConn`: prompt built from body+context, stream drained to text, output written.
- `NotebookCellRunner` dispatch: `spur` kernelspec → backend; `python` → unchanged runner (reuse engine.rs's `FakeRunner` + a new `FakeAiBackend`).
- Engine: stale-marking (upstream change marks stale, backend **not** called — assert call count 0); live mode cascades; cache hit skips the backend call.
- `PortStore` round-trip: consumed Arrow → rendered context; produced text port readable by a downstream cell.
- Follow the existing test style in the `engine.rs` `tests` module.

## 9. Tier-2 evolution hook (documented, not built)

`AiNodeBackend` is the seam. A future `OrchestratorSessionBackend` (multi-turn, tools, worker delegation via the `Orchestrator` / `spur-interactive` path analyzed earlier) becomes a second impl, selected by cell config. **Zero engine or cell-contract changes.**

## 10. Out of scope (YAGNI for Tier 1)

- Structured / schema output (→ Tier 1.5)
- Token-streaming into the cell UI (accumulate then write; can add via status events later)
- Multi-turn / session memory, tool use, worker delegation (→ Tier 2)
- The `Orchestrator` / `spur-interactive` integration
- Per-node model-selection UI
- Dedicated UI authoring beyond "kernelspec = `spur`" (the `jute-notebook/src/ui/dag` affordances are a separate follow-up)

## 11. Open input for the implementation plan

How the notebook obtains a **configured `AgentConnection`** — which adapter + agent binary/brain config. Reuse the same construction `spur-cli` / `spur-core` uses to spawn a brain. To be resolved during planning rather than guessed here.

---

*Next step after spec approval: invoke the writing-plans skill to produce the implementation plan (beads-backed DAG).*